<a href="https://colab.research.google.com/github/gchekwemoi/OOP-python-assignment-2/blob/main/Project2_uganda_tax_engine_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Mini-Project 2: Uganda Tax Engine
# Import the completed project from GitHub

!git clone https://github.com/gchekwemoi/OOP-python-assignment-2.git

%cd OOP-python-assignment-2

# Install project dependencies
!pip install -q -r requirements.txt

# Confirm the modules pass their automated tests
!python -m pytest tests/ -q


fatal: destination path 'OOP-python-assignment-2' already exists and is not an empty directory.
/content/OOP-python-assignment-2
........................................................................ [ 73%]
..........................                                               [100%]
98 passed in 1.53s


In [2]:
# Update the existing project from GitHub
%cd /content/OOP-python-assignment-2

!git pull origin main

# Run the latest automated tests
!python -m pytest tests/ -q

/content/OOP-python-assignment-2
From https://github.com/gchekwemoi/OOP-python-assignment-2
 * branch            main       -> FETCH_HEAD
Already up to date.
........................................................................ [ 73%]
..........................                                               [100%]
98 passed in 4.01s


In [3]:

# Payroll, NSSF and Annual LST Demonstration

from datetime import date
import pandas as pd

from src.tax_engine.money import Money
from src.tax_engine.payroll import PayrollCalculator

# Create payroll calculator
payroll = PayrollCalculator()

# Sample employees
employees = [
    {"name": "Employee A", "salary": 500000, "allowances": 0},
    {"name": "Employee B", "salary": 1000000, "allowances": 200000},
    {"name": "Employee C", "salary": 1500000, "allowances": 300000},
    {"name": "Employee D", "salary": 3000000, "allowances": 500000}
]

def payroll_table(month):
    """Calculate payroll for a selected month."""

    results = []

    for employee in employees:
        result = payroll.calculate(
            basic_salary=Money(employee["salary"]),
            allowances=Money(employee["allowances"]),
            residency="resident",
            tax_date=date(2026, month, 1)
        )

        results.append({
            "Employee": employee["name"],
            "Gross Pay": int(result.gross_pay.amount),
            "PAYE": int(result.paye.amount),
            "Employee NSSF": int(result.employee_nssf.amount),
            "Employer NSSF": int(result.employer_nssf.amount),
            "Annual LST": int(result.annual_local_service_tax.amount),
            "LST Deducted": int(result.local_service_tax.amount),
            "Net Pay": int(result.net_pay.amount)
        })

    return pd.DataFrame(results)


# July: Annual LST deducted in full
print("JULY 2026 PAYROLL - ANNUAL LST DEDUCTED")
july_df = payroll_table(7)

display(july_df.style.format({
    column: "{:,.0f}"
    for column in july_df.columns
    if column != "Employee"
}))


# October: No additional LST deduction
print("\nOCTOBER 2026 PAYROLL - LST ALREADY PAID")
october_df = payroll_table(10)

display(october_df.style.format({
    column: "{:,.0f}"
    for column in october_df.columns
    if column != "Employee"
}))


JULY 2026 PAYROLL - ANNUAL LST DEDUCTED


,Employee,Gross Pay,PAYE,Employee NSSF,Employer NSSF,Annual LST,LST Deducted,Net Pay
0,Employee A,"500,000","30,000","25,000","50,000","30,000","30,000","415,000"
1,Employee B,"1,200,000","221,250","60,000","120,000","90,000","90,000","828,750"
2,Employee C,"1,800,000","398,250","90,000","180,000","100,000","100,000","1,211,750"
3,Employee D,"3,500,000","908,250","175,000","350,000","100,000","100,000","2,316,750"



OCTOBER 2026 PAYROLL - LST ALREADY PAID


,Employee,Gross Pay,PAYE,Employee NSSF,Employer NSSF,Annual LST,LST Deducted,Net Pay
0,Employee A,"500,000","38,250","25,000","50,000","30,000",0,"436,750"
1,Employee B,"1,200,000","248,250","60,000","120,000","90,000",0,"891,750"
2,Employee C,"1,800,000","428,250","90,000","180,000","100,000",0,"1,281,750"
3,Employee D,"3,500,000","938,250","175,000","350,000","100,000",0,"2,386,750"


### Payroll, NSSF and Local Service Tax Results

The payroll simulation demonstrates the calculation of PAYE, NSSF contributions, Local Service Tax (LST) and net salary for four hypothetical employees earning different monthly salaries and allowances.

The results show that annual LST increases with the employee's income assessment band, ranging from UGX 30,000 to UGX 100,000 among the simulated employees. Under the model's assumption that the full annual LST is deducted once in July, total LST deductions amounted to UGX 320,000 for the four employees. No additional LST was deducted in October.

The July deductions also reduced the PAYE calculated for that month under the model's configured treatment of LST. Consequently, employees received lower net salaries in July than in October.

From an Object-Oriented Programming perspective, the simulation demonstrates **composition**, because `PayrollCalculator` reuses the `PAYETax` and `Money` classes; **encapsulation**, because payroll calculations are contained within the payroll class; and **code reuse**, since the same calculator processes multiple employees.

The results are illustrative and reflect the simplified assumptions used for this academic project.

In [6]:

# Step 24: Uganda VAT Calculation Demonstration

from datetime import date
import pandas as pd

from src.tax_engine.money import Money
from src.tax_engine.models import TaxContext
from src.tax_engine.vat import VATInvoice, VATTax

# Create sample business transactions
invoices = [
    VATInvoice(
        invoice_number="S001",
        invoice_type="sale",
        amount=Money("5000000"),
        supply_type="standard"
    ),
    VATInvoice(
        invoice_number="S002",
        invoice_type="sale",
        amount=Money("1000000"),
        supply_type="zero"
    ),
    VATInvoice(
        invoice_number="S003",
        invoice_type="sale",
        amount=Money("500000"),
        supply_type="exempt"
    ),
    VATInvoice(
        invoice_number="P001",
        invoice_type="purchase",
        amount=Money("2000000"),
        supply_type="standard",
        input_vat_eligible=True
    ),
    VATInvoice(
        invoice_number="P002",
        invoice_type="purchase",
        amount=Money("500000"),
        supply_type="standard",
        input_vat_eligible=False
    )
]

# Calculate VAT for each invoice
rows = []

for invoice in invoices:
    rows.append({
        "Invoice": invoice.invoice_number,
        "Type": invoice.invoice_type.title(),
        "Supply": invoice.supply_type.title(),
        "Amount (UGX)": int(invoice.amount.amount),
        "VAT (UGX)": int(invoice.vat_amount().amount),
        "Input VAT Eligible": (
            "Yes" if invoice.input_vat_eligible else "No"
        ) if invoice.invoice_type == "purchase" else "N/A"
    })

vat_df = pd.DataFrame(rows)

print("UGANDA VAT TRANSACTION SUMMARY")

display(vat_df.style.format({
    "Amount (UGX)": "{:,.0f}",
    "VAT (UGX)": "{:,.0f}"
}))

# Create VAT calculation context
vat_context = TaxContext(
    tax_date=date(2026, 10, 1),
    details={"invoices": invoices}
)

# Calculate total VAT
vat_result = VATTax().calculate(vat_context)

# Prepare summary
vat_summary = pd.DataFrame([
    {
        "Component": "Output VAT",
        "Amount (UGX)": int(
            vat_result.breakdown["output_vat"].amount
        )
    },
    {
        "Component": "Eligible Input VAT",
        "Amount (UGX)": int(
            vat_result.breakdown["input_vat"].amount
        )
    },
    {
        "Component": "Net VAT",
        "Amount (UGX)": int(
            vat_result.breakdown["net_vat"].amount
        )
    },
    {
        "Component": "VAT Payable",
        "Amount (UGX)": int(vat_result.amount.amount)
    }
])

print("\nVAT CALCULATION SUMMARY")

display(vat_summary.style.format({
    "Amount (UGX)": "{:,.0f}"
}))


UGANDA VAT TRANSACTION SUMMARY


,Invoice,Type,Supply,Amount (UGX),VAT (UGX),Input VAT Eligible
0,S001,Sale,Standard,"5,000,000","900,000",N/A
1,S002,Sale,Zero,"1,000,000",0,N/A
2,S003,Sale,Exempt,"500,000",0,N/A
3,P001,Purchase,Standard,"2,000,000","360,000",Yes
4,P002,Purchase,Standard,"500,000","90,000",No



VAT CALCULATION SUMMARY


,Component,Amount (UGX)
0,Output VAT,"900,000"
1,Eligible Input VAT,"360,000"
2,Net VAT,"540,000"
3,VAT Payable,"540,000"


### VAT Calculation Demonstration

This section demonstrates the use of Object-Oriented Programming to calculate Value Added Tax (VAT) on hypothetical business transactions in Uganda. The model applies the standard VAT rate of 18% and distinguishes between standard-rated, zero-rated and exempt supplies.

The `VATInvoice` class represents individual transactions, while the `VATTax` class processes the invoices and calculates output VAT, eligible input VAT and net VAT payable. For the simulated transactions, output VAT amounted to UGX 900,000 and eligible input VAT amounted to UGX 360,000, resulting in net VAT payable of UGX 540,000.

The example also demonstrates how input VAT can be excluded when a purchase is not eligible for deduction. The model assumes VAT-exclusive invoice values and that input VAT eligibility has already been determined.

From an Object-Oriented Programming perspective, the module demonstrates **inheritance**, because `VATTax` implements the abstract `Tax` interface; **encapsulation**, because VAT calculations are organised within classes; and **composition**, because the calculator processes a collection of `VATInvoice` objects.

In [7]:

# Step 25: Uganda Corporate Income Tax Demonstration

from datetime import date
import pandas as pd

from src.tax_engine.money import Money
from src.tax_engine.models import TaxContext
from src.tax_engine.corporate_tax import CorporateIncomeTax

# Create the Corporate Income Tax calculator
cit_calculator = CorporateIncomeTax()

# Hypothetical company scenarios
companies = [
    {
        "company": "Company A",
        "revenue": 250000000,
        "deductions": 150000000
    },
    {
        "company": "Company B",
        "revenue": 180000000,
        "deductions": 120000000
    },
    {
        "company": "Company C",
        "revenue": 100000000,
        "deductions": 100000000
    },
    {
        "company": "Company D",
        "revenue": 80000000,
        "deductions": 100000000
    }
]

results = []

for company in companies:

    context = TaxContext(
        tax_date=date(2026, 10, 1),
        income=Money(company["revenue"]),
        details={
            "allowable_deductions": Money(company["deductions"])
        }
    )

    tax_result = cit_calculator.calculate(context)

    results.append({
        "Company": company["company"],
        "Revenue": int(company["revenue"]),
        "Allowable Deductions": int(company["deductions"]),
        "Chargeable Income": int(
            tax_result.breakdown["chargeable_income"].amount
        ),
        "CIT Payable": int(tax_result.amount.amount),
        "Excess Deductions": int(
            tax_result.breakdown["excess_deductions"].amount
        )
    })

cit_df = pd.DataFrame(results)

print("UGANDA CORPORATE INCOME TAX CALCULATIONS")

display(
    cit_df.style.format({
        column: "{:,.0f}"
        for column in cit_df.columns
        if column != "Company"
    })
)


UGANDA CORPORATE INCOME TAX CALCULATIONS


,Company,Revenue,Allowable Deductions,Chargeable Income,CIT Payable,Excess Deductions
0,Company A,"250,000,000","150,000,000","100,000,000","30,000,000",0
1,Company B,"180,000,000","120,000,000","60,000,000","18,000,000",0
2,Company C,"100,000,000","100,000,000",0,0,0
3,Company D,"80,000,000","100,000,000",0,0,"20,000,000"


### Corporate Income Tax Demonstration

This section demonstrates the calculation of Corporate Income Tax (CIT) for four hypothetical companies in Uganda using the `CorporateIncomeTax` class.

The model calculates chargeable income by subtracting allowable deductions from annual revenue and applies the standard corporate income tax rate of 30%. Where deductions equal or exceed revenue, the model calculates zero current corporate income tax and reports excess deductions separately.

The results show that Company A has a CIT liability of UGX 30 million, while Company B has a liability of UGX 18 million. Company C has no chargeable income, and Company D reports UGX 20 million in excess deductions, resulting in zero current tax under the simplified model.

The demonstration illustrates **inheritance**, as `CorporateIncomeTax` implements the abstract `Tax` class; **encapsulation**, as the computation is contained within the class; and **code reuse**, through the common `Money`, `TaxContext` and `TaxResult` objects.

The model is designed for academic purposes and does not account for all statutory tax adjustments, exemptions, loss carry-forward provisions or alternative tax regimes.

In [8]:

# Step 26: TaxPlanner Scenario Comparison

from datetime import date
import pandas as pd

from src.tax_engine.money import Money
from src.tax_engine.models import TaxContext
from src.tax_engine.corporate_tax import CorporateIncomeTax
from src.tax_engine.paye import PAYETax
from src.tax_engine.planner import TaxPlanner

TAX_DATE = date(2026, 10, 1)

# SCENARIO 1: CORPORATE INCOME TAX

cit_planner = TaxPlanner(CorporateIncomeTax())

company_a = TaxContext(
    tax_date=TAX_DATE,
    income=Money("250000000"),
    details={
        "allowable_deductions": Money("150000000")
    }
)

company_b = TaxContext(
    tax_date=TAX_DATE,
    income=Money("250000000"),
    details={
        "allowable_deductions": Money("180000000")
    }
)

cit_comparison = cit_planner.compare(
    company_a, company_b
)

# SCENARIO 2: PAYE

paye_planner = TaxPlanner(PAYETax())

employee_a = TaxContext(
    tax_date=TAX_DATE,
    income=Money("1500000"),
    residency="resident"
)

employee_b = TaxContext(
    tax_date=TAX_DATE,
    income=Money("1000000"),
    residency="resident"
)

paye_comparison = paye_planner.compare(
    employee_a, employee_b
)

# Display the comparisons

comparison_df = pd.DataFrame([
    {
        "Tax Type": "Corporate Income Tax",
        "Scenario A Tax": int(
            cit_comparison.scenario_a.amount.amount
        ),
        "Scenario B Tax": int(
            cit_comparison.scenario_b.amount.amount
        ),
        "Difference (A - B)": int(
            cit_comparison.difference.amount
        ),
        "Lower Tax": cit_comparison.lower_tax_scenario
    },
    {
        "Tax Type": "PAYE",
        "Scenario A Tax": int(
            paye_comparison.scenario_a.amount.amount
        ),
        "Scenario B Tax": int(
            paye_comparison.scenario_b.amount.amount
        ),
        "Difference (A - B)": int(
            paye_comparison.difference.amount
        ),
        "Lower Tax": paye_comparison.lower_tax_scenario
    }
])

print("UGANDA TAXPLANNER - SCENARIO COMPARISON")

display(comparison_df.style.format({
    "Scenario A Tax": "{:,.0f}",
    "Scenario B Tax": "{:,.0f}",
    "Difference (A - B)": "{:,.0f}"
}))


UGANDA TAXPLANNER - SCENARIO COMPARISON


,Tax Type,Scenario A Tax,Scenario B Tax,Difference (A - B),Lower Tax
0,Corporate Income Tax,"30,000,000","21,000,000","9,000,000",Scenario B
1,PAYE,"338,250","188,250","150,000",Scenario B


### TaxPlanner: Comparing Alternative Tax Scenarios

The TaxPlanner extension demonstrates how Object-Oriented Programming can support tax scenario analysis without duplicating existing tax calculation methods.

Two comparisons were conducted. The first compared Corporate Income Tax liabilities under different levels of allowable deductions. Scenario A generated CIT of UGX 30 million, while Scenario B generated UGX 21 million, representing a difference of UGX 9 million.

The second comparison examined PAYE liabilities at two different monthly salary levels. The results showed PAYE of UGX 338,250 under Scenario A and UGX 188,250 under Scenario B, a difference of UGX 150,000.

The `TaxPlanner` class demonstrates **composition** by accepting an existing tax calculator object, and **polymorphism** by using the common `calculate()` interface implemented by different tax classes. Consequently, the same comparison method can work with both PAYE and Corporate Income Tax.

The scenarios are hypothetical and illustrate differences in calculated liabilities, not recommendations for avoiding or reducing legally payable tax.


In [10]:

# Step 28: Interactive Uganda Tax Calculator
# Install and import interactive controls

!pip install -q ipywidgets

import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import date

from src.tax_engine.money import Money
from src.tax_engine.payroll import PayrollCalculator

print("Interactive calculator components loaded successfully.")


Interactive calculator components loaded successfully.


In [11]:

# Interactive Uganda Payroll Calculator
# Users enter their own figures and click Calculate

# Input fields
salary_input = widgets.IntText(
    value=1500000,
    description="Basic Salary:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="350px")
)

allowances_input = widgets.IntText(
    value=0,
    description="Allowances:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="350px")
)

residency_input = widgets.Dropdown(
    options=[
        ("Resident", "resident"),
        ("Non-Resident", "non-resident")
    ],
    value="resident",
    description="Residency:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="350px")
)

month_input = widgets.Dropdown(
    options=[
        ("July", 7),
        ("August", 8),
        ("September", 9),
        ("October", 10),
        ("November", 11),
        ("December", 12),
        ("January", 1),
        ("February", 2),
        ("March", 3),
        ("April", 4),
        ("May", 5),
        ("June", 6)
    ],
    value=7,
    description="Payroll Month:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="350px")
)

lst_exempt_input = widgets.Checkbox(
    value=False,
    description="Employee is exempt from LST"
)

calculate_button = widgets.Button(
    description="Calculate Payroll",
    button_style="success",
    icon="calculator",
    layout=widgets.Layout(width="200px")
)

output = widgets.Output()


def calculate_payroll_clicked(button):
    """Calculate payroll using user-entered figures."""

    with output:
        clear_output(wait=True)

        try:
            # Select the correct calendar year
            month = month_input.value
            year = 2026 if month >= 7 else 2027

            calculator = PayrollCalculator()

            result = calculator.calculate(
                basic_salary=Money(salary_input.value),
                allowances=Money(allowances_input.value),
                residency=residency_input.value,
                tax_date=date(year, month, 1),
                lst_exempt=lst_exempt_input.value
            )

            print("=" * 45)
            print("UGANDA PAYROLL CALCULATION")
            print("=" * 45)

            print(f"Payroll Period: {month:02d}/{year}")
            print(f"Residency: {residency_input.value}")
            print("-" * 45)

            print(f"Gross Salary:     {result.gross_pay}")
            print(f"PAYE:             {result.paye}")
            print(f"Employee NSSF:    {result.employee_nssf}")
            print(f"Employer NSSF:    {result.employer_nssf}")
            print(f"Annual LST:       {result.annual_local_service_tax}")
            print(f"LST Deducted:     {result.local_service_tax}")

            print("-" * 45)
            print(f"NET SALARY:       {result.net_pay}")
            print("=" * 45)

        except (ValueError, TypeError) as error:
            print(f"Invalid input: {error}")

        except Exception as error:
            print(f"Calculation error: {error}")


calculate_button.on_click(calculate_payroll_clicked)

# Display interface
display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Uganda Interactive Payroll Calculator</h3>"
            "<p>Enter employee details and click Calculate.</p>"
        ),
        salary_input,
        allowances_input,
        residency_input,
        month_input,
        lst_exempt_input,
        calculate_button,
        output
    ])
)


In [12]:

# Step 29: Interactive Uganda VAT Calculator

import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import date

from src.tax_engine.money import Money
from src.tax_engine.models import TaxContext
from src.tax_engine.vat import VATInvoice, VATTax
from src.tax_engine.exceptions import TaxError

# Store invoices entered by the user
vat_invoices = []

# Input fields
invoice_number = widgets.Text(
    value="INV001",
    description="Invoice No:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="380px")
)

transaction_type = widgets.Dropdown(
    options=[
        ("Sale", "sale"),
        ("Purchase", "purchase")
    ],
    description="Transaction:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="380px")
)

invoice_amount = widgets.IntText(
    value=1000000,
    description="Amount (UGX):",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="380px")
)

supply_type = widgets.Dropdown(
    options=[
        ("Standard-rated (18%)", "standard"),
        ("Zero-rated (0%)", "zero"),
        ("Exempt", "exempt")
    ],
    description="VAT Treatment:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="380px")
)

input_eligible = widgets.Checkbox(
    value=True,
    description="Input VAT is eligible for deduction"
)

add_button = widgets.Button(
    description="Add Invoice",
    button_style="info",
    icon="plus"
)

calculate_vat_button = widgets.Button(
    description="Calculate VAT",
    button_style="success",
    icon="calculator"
)

clear_button = widgets.Button(
    description="Clear Invoices",
    button_style="warning",
    icon="trash"
)

vat_output = widgets.Output()


def update_eligibility(change=None):
    """Only show input VAT eligibility for purchases."""
    input_eligible.layout.display = (
        "flex" if transaction_type.value == "purchase" else "none"
    )


transaction_type.observe(
    update_eligibility,
    names="value"
)
update_eligibility()


def show_invoice_list():
    """Display invoices added by the user."""

    with vat_output:
        clear_output(wait=True)

        print("INVOICES ADDED")
        print("-" * 65)

        if not vat_invoices:
            print("No invoices added yet.")
            return

        for invoice in vat_invoices:
            print(
                f"{invoice.invoice_number} | "
                f"{invoice.invoice_type.title()} | "
                f"{invoice.supply_type.title()} | "
                f"{invoice.amount} | "
                f"VAT: {invoice.vat_amount()}"
            )

        print("-" * 65)
        print(f"Total invoices: {len(vat_invoices)}")


def add_invoice_clicked(button):
    """Add a new VAT invoice."""

    try:
        if any(
            inv.invoice_number == invoice_number.value.strip()
            for inv in vat_invoices
        ):
            raise ValueError("This invoice number already exists.")

        invoice = VATInvoice(
            invoice_number=invoice_number.value.strip(),
            invoice_type=transaction_type.value,
            amount=Money(invoice_amount.value),
            supply_type=supply_type.value,
            input_vat_eligible=(
                input_eligible.value
                if transaction_type.value == "purchase"
                else False
            )
        )

        vat_invoices.append(invoice)

        show_invoice_list()

        # Prepare the next invoice number
        invoice_number.value = f"INV{len(vat_invoices) + 1:03d}"

    except (ValueError, TypeError, TaxError) as error:
        with vat_output:
            clear_output(wait=True)
            print(f"Cannot add invoice: {error}")


def calculate_vat_clicked(button):
    """Calculate VAT using all added invoices."""

    with vat_output:
        clear_output(wait=True)

        if not vat_invoices:
            print("Please add at least one invoice first.")
            return

        try:
            context = TaxContext(
                tax_date=date(2026, 10, 1),
                details={"invoices": vat_invoices}
            )

            result = VATTax().calculate(context)

            print("=" * 50)
            print("UGANDA VAT CALCULATION RESULTS")
            print("=" * 50)

            print(f"Invoices processed: {len(vat_invoices)}")
            print("-" * 50)

            print(
                f"Output VAT:       "
                f"{result.breakdown['output_vat']}"
            )

            print(
                f"Eligible Input VAT: "
                f"{result.breakdown['input_vat']}"
            )

            print(
                f"Net VAT Position: "
                f"{result.breakdown['net_vat']}"
            )

            print("-" * 50)

            print(f"VAT PAYABLE:      {result.amount}")

            print(
                f"Excess Input VAT: "
                f"{result.breakdown['excess_input_vat']}"
            )

            print("=" * 50)

        except (ValueError, TypeError, TaxError) as error:
            print(f"Calculation error: {error}")


def clear_invoices_clicked(button):
    """Remove all invoices from the current calculation."""

    vat_invoices.clear()
    show_invoice_list()


# Connect buttons to their functions
add_button.on_click(add_invoice_clicked)
calculate_vat_button.on_click(calculate_vat_clicked)
clear_button.on_click(clear_invoices_clicked)


# Display the interface
display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Uganda Interactive VAT Calculator</h3>"
            "<p>Enter VAT-exclusive invoice amounts, "
            "add transactions and calculate VAT.</p>"
        ),
        invoice_number,
        transaction_type,
        invoice_amount,
        supply_type,
        input_eligible,
        widgets.HBox([
            add_button,
            calculate_vat_button,
            clear_button
        ]),
        vat_output
    ])
)


In [13]:

# Step 30: Interactive Uganda Corporate Income Tax Calculator

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import date

from src.tax_engine.money import Money
from src.tax_engine.models import TaxContext
from src.tax_engine.corporate_tax import CorporateIncomeTax

# Create input fields
cit_revenue = widgets.IntText(
    value=250000000,
    description="Annual Revenue:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="420px")
)

cit_deductions = widgets.IntText(
    value=150000000,
    description="Allowable Deductions:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="420px")
)

cit_button = widgets.Button(
    description="Calculate Corporate Tax",
    button_style="success",
    icon="calculator",
    layout=widgets.Layout(width="250px", height="40px")
)

cit_output = widgets.Output()


def calculate_cit_clicked(button):
    """Calculate CIT from user-entered figures."""

    with cit_output:
        clear_output(wait=True)

        try:
            revenue = Money(cit_revenue.value)
            deductions = Money(cit_deductions.value)

            context = TaxContext(
                tax_date=date(2026, 10, 1),
                income=revenue,
                details={
                    "allowable_deductions": deductions
                }
            )

            result = CorporateIncomeTax().calculate(context)

            chargeable_income = result.breakdown[
                "chargeable_income"
            ]

            excess_deductions = result.breakdown[
                "excess_deductions"
            ]

            print("=" * 52)
            print("UGANDA CORPORATE INCOME TAX CALCULATION")
            print("=" * 52)

            print(f"Annual Revenue:      {revenue}")
            print(f"Allowable Deductions: {deductions}")
            print("-" * 52)

            print(f"Chargeable Income:   {chargeable_income}")
            print("Corporate Tax Rate:  30%")

            print("-" * 52)
            print(f"CORPORATE TAX DUE:   {result.amount}")
            print("=" * 52)

            if excess_deductions.amount > 0:
                print(
                    f"\nExcess deductions: {excess_deductions}"
                )
                print(
                    "Note: This does not automatically "
                    "represent an approved tax loss."
                )

        except (ValueError, TypeError) as error:
            print(f"Invalid input: {error}")

        except Exception as error:
            print(f"Calculation error: {error}")


cit_button.on_click(calculate_cit_clicked)

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Uganda Corporate Income Tax Calculator</h3>"
            "<p>Enter annual revenue and allowable deductions "
            "to estimate corporate income tax.</p>"
        ),
        cit_revenue,
        cit_deductions,
        cit_button,
        cit_output
    ])
)


### Interactive Corporate Income Tax Calculator

The interactive Corporate Income Tax calculator allows users to enter annual revenue and allowable deductions without modifying the underlying Python code. The system calculates chargeable income and estimates corporate income tax using the configured standard rate of 30%.

The interface was implemented using `ipywidgets` and connects directly to the existing `CorporateIncomeTax` class. This separates user interaction from the underlying tax calculation logic, demonstrating encapsulation, modularity and code reuse.

The calculator is intended for educational demonstration and does not incorporate every statutory adjustment or special tax regime applicable in Uganda.

In [14]:

# Step 31: Interactive Uganda TaxPlanner

import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import date

from src.tax_engine.money import Money
from src.tax_engine.models import TaxContext
from src.tax_engine.paye import PAYETax
from src.tax_engine.corporate_tax import CorporateIncomeTax
from src.tax_engine.planner import TaxPlanner


# Select the tax to compare
planner_tax_type = widgets.Dropdown(
    options=[
        ("Corporate Income Tax", "cit"),
        ("PAYE", "paye")
    ],
    description="Tax Type:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

# Scenario inputs
planner_a_income = widgets.IntText(
    value=250000000,
    description="A: Revenue/Salary:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

planner_a_deductions = widgets.IntText(
    value=150000000,
    description="A: Deductions:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

planner_b_income = widgets.IntText(
    value=250000000,
    description="B: Revenue/Salary:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

planner_b_deductions = widgets.IntText(
    value=180000000,
    description="B: Deductions:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

planner_residency = widgets.Dropdown(
    options=[
        ("Resident", "resident"),
        ("Non-Resident", "non-resident")
    ],
    description="PAYE Residency:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

planner_button = widgets.Button(
    description="Compare Scenarios",
    button_style="success",
    icon="balance-scale",
    layout=widgets.Layout(width="230px", height="40px")
)

planner_output = widgets.Output()


def update_planner_fields(change=None):
    """Adapt the fields for the selected tax type."""

    is_cit = planner_tax_type.value == "cit"

    planner_a_deductions.layout.display = (
        "flex" if is_cit else "none"
    )

    planner_b_deductions.layout.display = (
        "flex" if is_cit else "none"
    )

    planner_residency.layout.display = (
        "none" if is_cit else "flex"
    )

    if is_cit:
        planner_a_income.description = "A: Annual Revenue:"
        planner_b_income.description = "B: Annual Revenue:"
    else:
        planner_a_income.description = "A: Monthly Salary:"
        planner_b_income.description = "B: Monthly Salary:"


planner_tax_type.observe(
    update_planner_fields,
    names="value"
)
update_planner_fields()


def compare_scenarios_clicked(button):
    """Compare user-entered tax scenarios."""

    with planner_output:
        clear_output(wait=True)

        try:
            tax_date = date(2026, 10, 1)

            if planner_tax_type.value == "cit":
                calculator = CorporateIncomeTax()

                context_a = TaxContext(
                    tax_date=tax_date,
                    income=Money(planner_a_income.value),
                    details={
                        "allowable_deductions": Money(
                            planner_a_deductions.value
                        )
                    }
                )

                context_b = TaxContext(
                    tax_date=tax_date,
                    income=Money(planner_b_income.value),
                    details={
                        "allowable_deductions": Money(
                            planner_b_deductions.value
                        )
                    }
                )

            else:
                calculator = PAYETax()

                context_a = TaxContext(
                    tax_date=tax_date,
                    income=Money(planner_a_income.value),
                    residency=planner_residency.value
                )

                context_b = TaxContext(
                    tax_date=tax_date,
                    income=Money(planner_b_income.value),
                    residency=planner_residency.value
                )

            planner = TaxPlanner(calculator)

            comparison = planner.compare(
                context_a,
                context_b
            )

            print("=" * 48)
            print("UGANDA TAXPLANNER RESULTS")
            print("=" * 48)

            print(
                f"Tax Type: "
                f"{'Corporate Income Tax' if planner_tax_type.value == 'cit' else 'PAYE'}"
            )

            print("-" * 48)
            print(f"Scenario A Tax: {comparison.scenario_a.amount}")
            print(f"Scenario B Tax: {comparison.scenario_b.amount}")
            print(f"Difference (A - B): {comparison.difference}")
            print("-" * 48)
            print(f"Lower Tax: {comparison.lower_tax_scenario}")
            print("=" * 48)

        except Exception as error:
            print(f"Calculation error: {error}")


planner_button.on_click(compare_scenarios_clicked)


display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Uganda Interactive TaxPlanner</h3>"
            "<p>Compare two hypothetical tax scenarios.</p>"
        ),
        planner_tax_type,
        widgets.HTML("<b>Scenario A</b>"),
        planner_a_income,
        planner_a_deductions,
        widgets.HTML("<b>Scenario B</b>"),
        planner_b_income,
        planner_b_deductions,
        planner_residency,
        planner_button,
        planner_output
    ])
)


### Interactive TaxPlanner Demonstration

The interactive TaxPlanner enables users to compare two hypothetical tax scenarios by entering their own figures. The interface supports both Corporate Income Tax and PAYE calculations, allowing users to examine how changes in revenue, allowable deductions or employment income affect calculated tax liabilities.

The interface was developed using `ipywidgets` and reuses the existing `TaxPlanner`, `PAYETax` and `CorporateIncomeTax` classes. This demonstrates polymorphism because the same comparison method can operate on different tax calculator objects through their shared `calculate()` interface. It also demonstrates composition and separation of the user interface from the underlying tax computation logic.

The results are intended for educational scenario analysis and should not be interpreted as tax planning advice or a substitute for statutory tax calculations.

## Technical Discussion and Object-Oriented Design

### 1. Why Decimal Was Used Instead of Float

Financial calculations require precision because small rounding errors can affect tax liabilities. Python's `float` type uses binary floating-point representation, which cannot represent some decimal values exactly. The `Money` class therefore uses `decimal.Decimal` for monetary calculations and applies explicit rounding to whole Uganda shillings. This makes the calculations more predictable and suitable for the educational tax engine.

### 2. Inheritance and Polymorphism

The system uses an abstract `Tax` class that defines the `calculate()` method. The `PAYETax`, `VATTax` and `CorporateIncomeTax` classes inherit from this abstract class and implement their respective tax calculations.

This design demonstrates polymorphism because different tax calculators can be accessed through the same method. For example, `TaxPlanner` can compare PAYE scenarios or corporate income tax scenarios without requiring separate comparison algorithms.

### 3. Composition and Code Reuse

Composition is demonstrated by the `PayrollCalculator`, which uses the existing `PAYETax` class to calculate monthly income tax. Similarly, `TaxPlanner` accepts a tax calculator object and uses its existing functionality to compare scenarios.

The `Money`, `TaxContext` and `TaxResult` classes are reused throughout the system. This reduces code duplication and makes the application easier to maintain.

### 4. Exception Handling and Validation

The application uses custom exceptions, including `TaxError`, `InvalidIncomeError` and `NoScheduleForDateError`, to handle invalid inputs and unsupported tax schedules.

Examples include negative income, negative allowances, invalid invoice classifications and requests for PAYE calculations outside the configured schedule dates. Exception handling makes failures easier to understand and prevents invalid inputs from silently producing misleading results.

### 5. Testing and Reliability

Automated tests were developed using `pytest` to verify monetary operations, PAYE brackets, payroll deductions, Local Service Tax, VAT, Corporate Income Tax and TaxPlanner comparisons.

The project also uses the `Hypothesis` library for property-based testing. These tests check general properties of PAYE calculations, including non-negative liabilities, monotonicity and tax amounts not exceeding the income being tested.

GitHub Actions was configured to run the automated test suite whenever changes were committed to the repository. The latest workflow completed successfully.

### 6. Limitations

The Uganda Tax Engine is a simplified educational application rather than a production-ready tax administration system. It uses configured tax schedules and does not automatically incorporate future legislative amendments.

The payroll component assumes annual Local Service Tax is deducted once in July. This is a project-specific payment assumption rather than a universal statutory collection rule. The VAT module assumes VAT-exclusive invoice amounts and predetermined input VAT eligibility. The corporate income tax module does not comprehensively model tax adjustments, loss carry-forwards, special regimes or exemptions.

These limitations would need to be addressed before the application could be deployed for real taxpayer or employer use.

### 7. Conclusion

The Uganda Tax Engine demonstrates how Object-Oriented Programming can be applied to practical taxation problems. Through abstraction, inheritance, composition, polymorphism and exception handling, the project combines several tax calculations into a modular and reusable Python application.

The TaxPlanner extension further demonstrates the flexibility of the design by allowing different tax scenarios to be compared using existing calculator classes.